# Lesson 03 - Агентные шаблоны проектирования

В этом уроке мы рассматриваем три основных шаблона проектирования для создания эффективных AI-агентов:

1. **Чёткие инструкции для агента** — создание точных, определяющих роль подсказок, которые направляют поведение агента  
2. **Структурированный вывод с использованием моделей Pydantic** — обеспечение возврата агентом предсказуемых, проверенных данных  
3. **Агенты с одной ответственностью** — проектирование сфокусированных агентов, каждый из которых хорошо выполняет одну задачу

Мы применим каждый шаблон к сценарию **рекомендации туристических направлений**, последовательно создавая систему, которая может предлагать направления, проверять их доступность и управлять логистикой.


## Настройка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Шаблон 1: Чёткие инструкции для агента

Самый эффективный шаблон также самый простой: написание чётких, подробных инструкций для вашего агента.

Хорошие инструкции определяют:
- **Кто** агент (персона и тон)
- **Что** он должен делать (пошаговые обязанности)
- **Как** он должен себя вести (ограничения и стиль)

Ниже мы создаём туристического консьержа с явными инструкциями, которые формируют каждый ответ, который он даёт.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Шаблон 2: Структурированный вывод с моделями Pydantic

Свободный текст полезен для общения, но последующим системам нужны структурированные данные.
Комбинируя **модели Pydantic** с **функцией инструмента**, мы можем:

- Определить точную схему для вывода агента
- Автоматически проверять ответы
- Надежно интегрировать результаты агента в логику приложения

Мы также представляем инструмент, который возвращает детали назначения, чтобы агент мог основывать свои рекомендации на реальных данных.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Паттерн 3: Агенты с единой ответственностью

Сложные задачи выигрывают от разделения работы между несколькими специализированными агентами, каждый из которых отвечает за свою область:

- **Эксперт по направлению**, который знает о местах и наличии
- **Планировщик логистики**, который занимается рейсами, отелями и маршрутами

Это отражает принцип программной инженерии *разделения ответственности* — каждый агент проще тестировать, поддерживать и улучшать независимо.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Краткое содержание

В этом уроке мы применили три паттерна агентного дизайна к сценарию рекомендателя путешествий:

| Паттерн | Ключевая идея | Преимущество |
|---|---|---|
| **Чёткие инструкции** | Заранее определить персону, обязанности и ограничения | Последовательное поведение агента в соответствии с брендом |
| **Структурированный вывод** | Использовать модели Pydantic в качестве формата ответа | Проверенные, удобные для машинного чтения результаты |
| **Единая ответственность** | Назначить агенту одну конкретную задачу | Проще тестировать, поддерживать и комбинировать |

Эти паттерны естественно сочетаются — вы можете комбинировать чёткие инструкции со структурированным выводом в агенте с единой ответственностью для создания надёжных систем, готовых к производству.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
